In [ ]:
!pip install -q --upgrade --force-reinstall torch==2.10.0+cu128 torchvision==0.25.0+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip install -q transformers datasets peft trl accelerate
!pip install -q "torchao==0.10.0" "peft==0.14.0"
!git clone https://github.com/ggerganov/llama.cpp 2>/dev/null || true
!pip install -q -r llama.cpp/requirements.txt

In [ ]:
import os
import sys
import torch
import subprocess

from google.colab import drive
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

drive.mount('/content/drive')

In [ ]:
# ======================
# -- PATHS
# ======================

MODEL_ID   = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATASET_ID = "g0ofycatz/OkinDataset"

BASE_DIR   = "/content/drive/MyDrive/okin"
OUTPUT_DIR = os.path.join(BASE_DIR, "okin_llm")
GGUF_DIR   = os.path.join(BASE_DIR, "gguf")
LLAMA_CPP  = "/content/llama.cpp"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

# ======================
# -- DATA
# ======================

dataset   = load_dataset(DATASET_ID, split="train")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# ======================
# -- MODEL
# ======================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

# ======================
# -- TRAIN
# ======================

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_pin_memory=True,
    auto_find_batch_size=True,
    assistant_only_loss=True,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    warmup_steps=10,
    lr_scheduler_type="cosine",
    max_length=512,
    eos_token="<|im_end|>",
    optim="adamw_torch_fused",
)

trainer = SFTTrainer(
    model=model,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=args,
    train_dataset=dataset,
)

trainer.train()

merged = trainer.model.merge_and_unload()
merged.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# ======================
# -- GGUF
# ======================

convert   = os.path.join(LLAMA_CPP, "convert_hf_to_gguf.py")
gguf_fp16 = os.path.join(GGUF_DIR, "okin-qwen-fp16.gguf")

print("Converting to GGUF...")
subprocess.run(
    [sys.executable, convert, OUTPUT_DIR, "--outfile", gguf_fp16, "--outtype", "f16"],
    check=True,
)

print(f"Done: {gguf_fp16}")